# **vae-3d-spatio-spectral — inference evaluation**

Loads trained **vae-3d-spatio-spectral** checkpoints from `model/<DATASET>/` and evaluates reconstruction quality (MSE, SAM, PSNR, SSIM) on the test split, plus reconstruction visualizations. Self-contained (Kaggle/Colab-ready).

## **Config**

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path as _Path
import yaml as _yaml

# --------------------------------------------------------------------------
# Datasets in the ablation. Edit DATA_ROOTS to point at your processed patches
# (each root should contain <scene>/<split>/patch_*.npy). On Kaggle/Colab these
# will be /kaggle/input/... paths; locally they default to data/processed/<DS>.
# --------------------------------------------------------------------------
DATASETS = {
    "IIRS":   {"input_channels": 256},
    "M3":     {"input_channels": 84},
    "AVIRIS": {"input_channels": 424},
    "CRIMS":  {"input_channels": 544},
}

DATA_ROOTS = {
    "IIRS":   "data/processed/IIRS",
    "M3":     "data/processed/M3",
    "AVIRIS": "data/processed/AVIRIS",
    "CRIMS":  "data/processed/crims",
}

# Where trained checkpoints are written: CKPT_ROOT/<DATASET>/<name>.pt
CKPT_ROOT = "model"

# Where the per-dataset hyperparam YAMLs live (repo-root-relative).
HYPERPARAM_CONFIG_DIR = "utils/hyperparam_configs"


@dataclass
class Settings:
    input_height: int = 64
    input_width: int = 64
    input_channels: int = 256          # overridden per dataset via make_settings()

    # training
    batch_size: int = 4
    num_workers: int = 4
    epochs: int = 20
    lr: float = 5e-3
    beta: float = 1e-3
    lambda_physics: float = 0.3

    # spatial branch
    reduced_dims: int = 32
    latent_dim: int = 256
    n_2D_conv_blocks: int = 4
    conv2D_kernel_size: int = 3
    conv_output_c: int = field(init=False)
    conv_output_h: int = field(init=False)
    conv_output_w: int = field(init=False)

    # spectral branch
    spectral_n_1D_conv_blocks: int = 2
    spectral_conv1D_kernel_size: int = 4
    spectral_latent_dim: int = 128
    spectral_linear_expansion_dim: int = field(init=False)
    spectral_transpose_c: int = field(init=False)
    spectral_transpose_l: int = field(init=False)

    # Baseline capacity knobs (overridden per-dataset by hyperparam YAML so each
    # baseline matches vae-our's param count at that dataset). IIRS defaults.
    vae_standard_base_ch: int = 134
    vae_standard_n_down: int = 3
    vae_standard_latent_ch: int = 16

    vae_3d_base_ch: int = 78
    vae_3d_n_down: int = 3
    vae_3d_latent_ch: int = 8

    vae_1d_hidden_dims: tuple = (4224, 2112, 1056)
    vae_1d_latent_dim: int = 32

    def __post_init__(self):
        self.conv_output_c = self.reduced_dims * (2 ** self.n_2D_conv_blocks)
        self.conv_output_h = self.input_height // (2 ** self.n_2D_conv_blocks)
        self.conv_output_w = self.input_width // (2 ** self.n_2D_conv_blocks)
        self.spectral_transpose_c = self.input_channels * (2 ** (self.spectral_n_1D_conv_blocks - 1))
        self.spectral_transpose_l = self.input_channels // (2 ** self.spectral_n_1D_conv_blocks)
        self.spectral_linear_expansion_dim = self.spectral_transpose_c * self.spectral_transpose_l


def make_settings(dataset):
    """Return a Settings whose band count matches the dataset."""
    return Settings(input_channels=DATASETS[dataset]["input_channels"])


# --------------------------------------------------------------------------
# Per-dataset hyperparam loader — mirrors utils/hyperparams.py but re-implemented
# locally so the notebook stays self-contained.
# --------------------------------------------------------------------------
_HP_SETTINGS_FIELDS = {
    "batch_size", "num_workers",
    "vae_standard_base_ch", "vae_standard_n_down", "vae_standard_latent_ch",
    "vae_3d_base_ch", "vae_3d_n_down", "vae_3d_latent_ch",
    "vae_1d_hidden_dims", "vae_1d_latent_dim",
    # notebook Settings also carries these as fields (unlike utils/config.py Settings):
    "epochs", "lr", "beta", "lambda_physics",
}
_HP_OPTIMIZATION_FIELDS = {"seed", "weight_decay", "early_stopping_patience"}
_HP_ALLOWED = _HP_SETTINGS_FIELDS | _HP_OPTIMIZATION_FIELDS


def load_hyperparams(dataset, config_dir=HYPERPARAM_CONFIG_DIR):
    """Load hyperparam-config-<DATASET>.yaml -> dict. Empty dict if missing."""
    path = _Path(config_dir) / f"hyperparam-config-{dataset.upper()}.yaml"
    if not path.exists():
        return {}
    with open(path) as f:
        raw = _yaml.safe_load(f) or {}
    unknown = set(raw) - _HP_ALLOWED
    if unknown:
        raise ValueError(f"{path}: unknown hyperparam keys: {sorted(unknown)}")
    return raw


def apply_hyperparams(settings, hp):
    """Mutate `settings` in place for whitelisted fields present in `hp`."""
    for key in _HP_SETTINGS_FIELDS & set(hp):
        value = hp[key]
        if key == "vae_1d_hidden_dims" and isinstance(value, list):
            value = tuple(value)
        setattr(settings, key, value)


# Global settings object the branch classes read from. Reassigned per dataset
# inside the training loop (rebuild the model after reassigning).
settings = make_settings("IIRS")


## **Imports**

In [ ]:
import math
import time
from pathlib import Path
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch import Tensor
from torch.nn import Conv2d, ConvTranspose2d
from torch.utils.data import Dataset, DataLoader

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())


## **Loss and metrics**

In [ ]:
def spectral_angle_mapper_loss(y_true, y_pred):
    """Differentiable physics prior: mean spectral angle (radians)."""
    dot = torch.sum(y_true * y_pred, dim=-1)
    nt = torch.sqrt(torch.sum(y_true ** 2, dim=-1) + 1e-8)
    npd = torch.sqrt(torch.sum(y_pred ** 2, dim=-1) + 1e-8)
    cos = torch.clamp(dot / (nt * npd + 1e-8), -1.0 + 1e-8, 1.0 - 1e-8)
    return torch.mean(torch.acos(cos))

In [ ]:
def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

In [ ]:
def compute_psnr(img1, img2, data_range=1.0):
    mse = F.mse_loss(img1, img2)
    if mse == 0:
        return float("inf")
    return (20 * torch.log10(torch.tensor(data_range).to(img1.device)) - 10 * torch.log10(mse)).item()

In [ ]:
def compute_ssim(img1, img2, data_range=1.0, window_size=11):
    if img1.dim() == 4 and img1.shape[-1] not in [img1.shape[1], img1.shape[2]]:
        img1 = img1.permute(0, 3, 1, 2); img2 = img2.permute(0, 3, 1, 2)
    channels = img1.shape[1]

    def gaussian(w, sigma):
        g = torch.exp(torch.tensor([-(x - w // 2) ** 2 / (2 * sigma ** 2) for x in range(w)]))
        return g / g.sum()

    _1d = gaussian(window_size, 1.5).unsqueeze(1).to(img1.device)
    _2d = _1d.mm(_1d.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2d.expand(channels, 1, window_size, window_size).contiguous()
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channels)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channels)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    s1 = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channels) - mu1_sq
    s2 = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channels) - mu2_sq
    s12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channels) - mu1_mu2
    c1 = (0.01 * data_range) ** 2; c2 = (0.03 * data_range) ** 2
    ssim = ((2 * mu1_mu2 + c1) * (2 * s12 + c2)) / ((mu1_sq + mu2_sq + c1) * (s1 + s2 + c2))
    return ssim.mean().item()


## **Data loader**

In [ ]:
class HSIPatchDataset(Dataset):
    """All .npy patches for a split, max-normalized to [0, 1] on the fly."""
    def __init__(self, processed_root, split):
        assert split in ("train", "valid", "test")
        self.patch_files: List[Path] = sorted(Path(processed_root).glob(f"**/{split}/*.npy"))

    def __len__(self):
        return len(self.patch_files)

    def __getitem__(self, idx):
        patch = np.load(self.patch_files[idx], mmap_mode="r").astype(np.float32)
        m = patch.max()
        if m > 0:
            patch = patch / m
        return torch.from_numpy(patch)


def build_dataloader(processed_root, split, batch_size=None, shuffle=True, num_workers=None, pin_memory=True):
    ds = HSIPatchDataset(processed_root, split)
    return DataLoader(
        ds,
        batch_size=batch_size or settings.batch_size,
        shuffle=shuffle,
        num_workers=num_workers if num_workers is not None else settings.num_workers,
        pin_memory=pin_memory,
        drop_last=(split == "train"),
    )


## **Model definition**

MODEL DEFINITION  --  vae-3d-spatio-spectral  (Baseline B: 3D Spatio-Spectral)

Fully-Conv3D VAE (Chen et al., 2016 / Palsson et al., 2018): the patch is a
single-channel volume (B, 1, C, H, W). 3D kernels average local bands with
local pixels; parameter-heavy and prone to posterior collapse. Spatial dims
are strided (H,W -> /8); spectral depth C is preserved (stride 1) so the
round-trip is exact for any band count.

In [ ]:
_K, _S, _P = (3, 4, 4), (1, 2, 2), (1, 1, 1)


class VAE_3D_SpatioSpectral(nn.Module):
    def __init__(self):
        super().__init__()
        base_ch = settings.vae_3d_base_ch
        n_down = settings.vae_3d_n_down
        latent_ch = settings.vae_3d_latent_ch
        self.latent_ch = latent_ch
        enc = [nn.Conv3d(1, base_ch, 3, 1, 1), nn.ReLU()]
        in_c = base_ch
        for _ in range(n_down):
            enc += [nn.Conv3d(in_c, in_c * 2, _K, _S, _P), nn.ReLU()]; in_c *= 2
        enc.append(nn.Conv3d(in_c, 2 * latent_ch, 1))
        self.encoder = nn.Sequential(*enc)
        dec = [nn.Conv3d(latent_ch, in_c, 1), nn.ReLU()]
        for _ in range(n_down):
            dec += [nn.ConvTranspose3d(in_c, in_c // 2, _K, _S, _P), nn.ReLU()]; in_c //= 2
        dec.append(nn.Conv3d(in_c, 1, 3, 1, 1))
        self.decoder = nn.Sequential(*dec)

    @staticmethod
    def reparameterize(params):
        mu, logvar = torch.chunk(params, 2, dim=1)
        logvar = torch.clamp(logvar, -30.0, 20.0)
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std, mu, logvar

    def forward(self, x):
        vol = x.permute(0, 3, 1, 2).unsqueeze(1)
        z, mu, logvar = self.reparameterize(self.encoder(vol))
        recon = torch.sigmoid(self.decoder(z)).squeeze(1).permute(0, 2, 3, 1)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_latents(self, x):
        mu, _ = torch.chunk(self.encoder(x.permute(0, 3, 1, 2).unsqueeze(1)), 2, dim=1)
        return [mu]

    @torch.no_grad()
    def decode_latents(self, latents):
        return torch.sigmoid(self.decoder(latents[0])).squeeze(1).permute(0, 2, 3, 1)


## **Ckpt helpers**

In [ ]:
from collections import OrderedDict
import matplotlib.pyplot as plt


def strip_module_prefix(state):
    out = OrderedDict()
    for k, v in state.items():
        out[k[len("module."):] if k.startswith("module.") else k] = v
    return out


def load_checkpoint(ckpt_file, device):
    model = build_model().to(device)
    with torch.no_grad():   # materialize LazyLinear
        model(torch.randn(2, settings.input_height, settings.input_width, settings.input_channels, device=device))
    ckpt = torch.load(ckpt_file, map_location=device)
    state = ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(strip_module_prefix(state))
    model.eval()
    return model


@torch.no_grad()
def reconstruct(model, x):
    out = model(x)
    return out[0] if isinstance(out, (tuple, list)) else out


## **Evaluate all checkpoints on the test split**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "vae-3d-spatio-spectral"

def ckpt_name(loss_type):
    return "vae-our.pt" if MODEL_NAME == "vae-our" else f"{MODEL_NAME}_{loss_type}.pt"

rows = []
for ds in ["IIRS", "M3", "AVIRIS", "CRIMS"]:
    for loss_type in ['standard', 'physics']:
        settings = make_settings(ds)
        hp = load_hyperparams(ds)
        apply_hyperparams(settings, hp)
        globals()["settings"] = settings
        ckpt_file = Path(CKPT_ROOT) / ds / ckpt_name(loss_type)
        if not ckpt_file.exists():
            print(f"[skip] missing {ckpt_file}"); continue

        model = load_checkpoint(ckpt_file, device)
        loader = build_dataloader(DATA_ROOTS[ds], "test", shuffle=False)

        mse = sam = psnr = ssim = 0.0; n = 0
        with torch.no_grad():
            for x in loader:
                x = x.to(device); recon = reconstruct(model, x)
                mse += F.mse_loss(x, recon).item()
                sam += spectral_angle_mapper_loss(x, recon).item()
                psnr += compute_psnr(x, recon); ssim += compute_ssim(x, recon); n += 1
        n = max(n, 1)
        rows.append((ds, loss_type, mse/n, sam/n, psnr/n, ssim/n))
        print(f"{ds:6s} {loss_type:8s} MSE {mse/n:.5f} SAM {sam/n:.4f} PSNR {psnr/n:.2f} SSIM {ssim/n:.4f}")

print("\n=== summary ===")
print(f"{'dataset':7s}{'loss':9s}{'MSE':>10s}{'SAM':>9s}{'PSNR':>9s}{'SSIM':>8s}")
for ds, lt, m, s, p, ss in rows:
    print(f"{ds:7s}{lt:9s}{m:10.5f}{s:9.4f}{p:9.2f}{ss:8.4f}")


## **Reconstruction visualizations**

In [ ]:
# Visualize reconstructions for one dataset/loss. Edit DS / LOSS / bands / pixel.
DS = "IIRS"; LOSS = "physics"; RGB_BANDS = (30, 20, 10); PIXEL = (32, 32); N = 4

settings = make_settings(DS); globals()["settings"] = settings
ckpt_file = Path(CKPT_ROOT) / DS / (("vae-our.pt") if MODEL_NAME == "vae-our" else f"{MODEL_NAME}_{LOSS}.pt")
model = load_checkpoint(ckpt_file, device)
loader = build_dataloader(DATA_ROOTS[DS], "test", batch_size=N, shuffle=False)
x = next(iter(loader)).to(device)
recon = reconstruct(model, x).cpu()
x = x.cpu()

fig, axes = plt.subplots(2, N, figsize=(4 * N, 8))
for i in range(N):
    gt = x[i][..., list(RGB_BANDS)].numpy(); gt = (gt - gt.min()) / (gt.ptp() + 1e-8)
    rc = recon[i][..., list(RGB_BANDS)].numpy(); rc = (rc - rc.min()) / (rc.ptp() + 1e-8)
    axes[0, i].imshow(gt); axes[0, i].set_title(f"GT {i}"); axes[0, i].axis("off")
    axes[1, i].imshow(rc); axes[1, i].set_title(f"Recon {i}"); axes[1, i].axis("off")
plt.suptitle(f"{MODEL_NAME} | {DS} | {LOSS} — pseudo-RGB {RGB_BANDS}"); plt.tight_layout(); plt.show()

# Spectral reflectance curve at one pixel
r, c = PIXEL
plt.figure(figsize=(8, 4))
plt.plot(x[0, r, c].numpy(), label="Ground truth")
plt.plot(recon[0, r, c].numpy(), label="Reconstruction", linestyle="--")
plt.xlabel("Band"); plt.ylabel("Reflectance (norm.)"); plt.legend()
plt.title(f"{MODEL_NAME} | {DS} | pixel {PIXEL}"); plt.show()
